# ParkingBill

[Codility task](https://app.codility.com/programmers/trainings/5/parking_bill/) - Exercise 5, Coding skills - *elementary*

Billing rules:

- entrance fee: `2`;
- the first full or partial hour: `3`;
- each following full or partial hour: `4`.

`solution(E, L)` receives entry and exit times as `"HH:MM"` strings (same day, `E` before `L`) and returns the total cost.

| E | L | result |
|---|---|--------|
| `10:00` | `13:21` | 17 (= 2 + 3 + 3 * 4) |
| `09:42` | `11:42` | 9 (= 2 + 3 + 4) |

## Idea

1. Convert both times to minutes since midnight.
2. Billed hours = **ceiling** of `minutes / 60`. Integer trick: `-(-minutes // 60)`.
3. Cost = entrance fee + first hour + `(hours - 1)` more hours.

Because `E < L`, the stay is at least one minute and therefore at least one hour is billed.

In [ ]:
ENTRANCE_FEE = 2
FIRST_HOUR = 3
NEXT_HOUR = 4


def to_minutes(hhmm):
    hours, minutes = hhmm.split(":")
    return int(hours) * 60 + int(minutes)


def solution(E, L):
    stay = to_minutes(L) - to_minutes(E)
    billed_hours = -(-stay // 60)                     # ceiling division
    return ENTRANCE_FEE + FIRST_HOUR + NEXT_HOUR * (billed_hours - 1)

## Official examples

In [ ]:
assert solution("10:00", "13:21") == 17
assert solution("09:42", "11:42") == 9
print("examples OK")

## Edge cases

The interesting spots are the hour boundaries: 59, 60 and 61 minutes, and the extremes of the day.

In [ ]:
assert solution("12:00", "12:01") == 5                # 1 minute  -> 1 hour
assert solution("12:00", "12:59") == 5                # 59 min    -> 1 hour
assert solution("12:00", "13:00") == 5                # exactly 60 -> still 1 hour
assert solution("12:00", "13:01") == 9                # 61 min    -> 2 hours
assert solution("12:00", "14:00") == 9                # exactly 120 -> 2 hours
assert solution("00:00", "23:59") == 2 + 3 + 4 * 23   # 1439 min  -> 24 hours = 97
assert solution("23:58", "23:59") == 5                # last minute of the day
assert solution("00:00", "00:01") == 5                # first minute of the day
assert solution("09:59", "10:00") == 5                # crossing an hour on the clock is irrelevant
print("edge cases OK")

## Reference check

An independent **simulation**: keep adding the price of one more hour until the stay is covered. It is deliberately naive and shares no formula with the solution. It is compared on many `(E, L)` pairs, including every possible stay length starting at midnight.

In [ ]:
def simulate(E, L):
    stay = to_minutes(L) - to_minutes(E)
    cost = ENTRANCE_FEE
    covered = 0
    first = True
    while covered < stay:
        cost += FIRST_HOUR if first else NEXT_HOUR
        first = False
        covered += 60
    return cost


def fmt(minutes):
    return f"{minutes // 60:02d}:{minutes % 60:02d}"


checked = 0
for start in list(range(0, 1440, 17)) + [1438, 1439]:
    for end in range(start + 1, 1440):
        assert solution(fmt(start), fmt(end)) == simulate(fmt(start), fmt(end)), (fmt(start), fmt(end))
        checked += 1
print(f"matches the simulation on {checked} (E, L) pairs")

## Notes

- Ceiling division without `math.ceil`: `-(-a // b)` or `(a + b - 1) // b`. Both avoid floating point.
- Do not compare the times as strings to compute a difference; compare minutes. (Comparing zero-padded `"HH:MM"` strings lexicographically does work for ordering, but not for subtraction.)
- The task guarantees `E < L` on the same day, so no midnight wrap-around handling is needed.